In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import random
import json
import re
from sklearn.decomposition import PCA
from collections import Counter
from datetime import datetime

save_path = "../data/"

if not os.path.exists(save_path + "plot/"):
        os.makedirs(save_path + "plot/")


## Specifications

In [ ]:
models = ["bert-base-uncased","microsoft-deberta-base"]

layers = [0, 3, 6, 9, 12]

datasets = ["semcor&omsti_noun-synsets", "semcor&omsti_person.n.01", "cctweets-activist", "cctweets-random"]
dataset =  "semcor&omsti_person.n.01"

## PCA plot settings

In [ ]:
def PCA_transform(matrix):
    pca = PCA(n_components = 2)
    pca.fit(matrix)
    transformed = pca.transform(matrix)
    return transformed, np.around(pca.explained_variance_ratio_*100,1)


tab20b = list(plt.cm.get_cmap("tab20b").colors)
tab20c = list(plt.cm.get_cmap("tab20c").colors)

color_dict = {
    0: tab20b[0],  #dark blue
    1: tab20b[2],  #mild blue
    2: tab20b[5], #darker green
    3: tab20b[7], #lighter green
    4: tab20b[1], 
    5: tab20b[3], 
    6: tab20b[4], 
    7: tab20b[6], 
    8: tab20b[8], 
    9: tab20b[9],
    10: tab20b[10],
    11: tab20b[11],
    12: tab20b[12],
    13: tab20b[13],
    14: tab20b[14],
    15: tab20b[15],
    16: tab20b[16],
    17: tab20b[17],
    18: tab20b[18],
    19: tab20b[19],
    20: tab20c[0],
    21: tab20c[1],
    22: tab20c[2],
    23: tab20c[3],
    24: tab20c[4],
    25: tab20c[5],
    26: tab20c[6]
    
}

for i in range(len(color_dict), 300):
    color_dict[i] = tab20c[17]

# Plot with positional information

In [ ]:
models = ["bert-base-uncased", "microsoft-deberta-base"] # splitted in two plots
layers = [0, 3, 6, 9, 12]  #[0, 3, 6, 9, 12]

datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", "cctweets-random", "cctweets-activist"]

plot_rows = len(layers)
plot_cols = len(datasets)


plt.rc('axes', labelsize=12)
plt.rc('axes', titlesize=12)  
plt.rc("font", size = 10)



for model in models:
      
    png, png_axs = plt.subplots(plot_rows,plot_cols, figsize = (3.5*plot_cols,3.5*plot_rows))
    
    plot_col = 0
    
    ### cols
    for dataset in datasets:
        
        try:
            dataset_name, filter_name = dataset.split("_")
        except:
            dataset_name = dataset
            filter_name = "None"
            
        plot_row = 0

        df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
        
            
        filter_mask = np.arange(0, len(df)) if filter_name == "None" else np.where(df[filter_name])
        df = df.loc[df[filter_name]== True] if filter_name != "None" else df
            
        
        ### rows
        for layer in layers:
            
            
            # loading and filtering embeddings
            embedding_file =  [file for file in os.listdir(f'{save_path}embedding/') if re.match("_".join((dataset_name, model, f"layer{layer}")), file)][0]
            embeddings = np.load(f'{save_path}embedding/{embedding_file}')[filter_mask]
            
            
            # loading the cluster labels
            try:
                clustering_file = [file for file in os.listdir(f'{save_path}clustering/') if re.match("_".join((dataset_name, model, f"layer{layer}", filter_name, "best")), file)][0]
                labels = np.load(f'{save_path}clustering/{clustering_file}')
            except:
                labels = [0 for i in range(len(embeddings))]
            
            color = df["relative_position"]
            
            s = 0.0005 if len(labels)> 10000 else 0.05
            
            
            pca, explained_variance_ratio_ = PCA_transform(embeddings)
            
            
            
            
            # scatter plot
            ax = png_axs[plot_row, plot_col].scatter(pca[:,0], pca[:,1], s=s, c = color, cmap = "magma")
            
            xlim1, xlim2 = png_axs[plot_row, plot_col].get_xlim()
            ylim1, ylim2 = png_axs[plot_row, plot_col].get_ylim()

            xlim1min = min(xlim1min, xlim1)
            xlim2max = max(xlim2max, xlim2)
            ylim1min = min(ylim1min, ylim1)
            ylim2max = max(ylim2max, ylim2)
            
            
            if plot_row == 0:
                
                title_filter_name = "noun-synsets" if filter_name == "noun-synsets-strat" else filter_name
                title = " ".join((dataset_name, title_filter_name)) if title_filter_name != "None" else dataset_name
                png_axs[plot_row, plot_col].set_title(title, fontfamily= "serif")
            
            if plot_col == 0:
                png_axs[plot_row, plot_col].set_ylabel(f"layer {layer}", fontfamily= "serif")
            
            k = max(labels)+1
            png_axs[plot_row, plot_col].text(0.81,0.04, f"k={k}", 
                                             transform = png_axs[plot_row,plot_col].transAxes)
            
            plot_row += 1
            
            
            
                
        # adjusting ax limits to be identical per dataset
        for plot_row in range(0,plot_rows):
            png_axs[plot_row, plot_col].set_xlim(xlim1min, xlim2max)
            png_axs[plot_row, plot_col].set_ylim(ylim1min, ylim2max)
        
        plot_col += 1
    

    png.subplots_adjust(right = 0.95)
    cbar_ax =png.add_axes([0.97, 0.11, 0.008,0.78])
    png.colorbar(ax, cax=cbar_ax)
    
        
    time = datetime.now().strftime("%m-%d-%H-%M")
    png.savefig(f'{save_path}plot/{model}_{time}.png', bbox_inches = "tight")
    plt.show()

        

## Cluster Assignment Plot with k = 2

In [ ]:
models = ["bert-base-uncased", "microsoft-deberta-base"] # splitted into two plots
layers = [0, 3, 6, 9, 12]  

datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", "cctweets-random", "cctweets-activist"]

plot_rows = len(layers)
plot_cols = len(datasets)

kmeans = "kmeans2"


plt.rc('axes', labelsize=12)
plt.rc('axes', titlesize=12)  
plt.rc("font", size = 10)



for model in models:
      
    png, png_axs = plt.subplots(plot_rows,plot_cols, figsize = (3.5*plot_cols,3.5*plot_rows))
    
    plot_col = 0
    
    ### cols
    for dataset in datasets:
        
        try:
            dataset_name, filter_name = dataset.split("_")
        except:
            dataset_name = dataset
            filter_name = "None"
            
        plot_row = 0
        
        df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
        
            
        filter_mask = np.arange(0, len(df)) if filter_name == "None" else np.where(df[filter_name])
            
        
        ### rows
        for layer in layers:
            
            
            # loading and filtering embeddings
            embedding_file =  [file for file in os.listdir(f'{save_path}embedding/') if re.match("_".join((dataset_name, model, f"layer{layer}")), file)][0]
            embeddings = np.load(f'{save_path}embedding/{embedding_file}')[filter_mask]
            
            
            # loading the cluster labels
            try:
                clustering_file = [file for file in os.listdir(f'{save_path}clustering/') if re.match("_".join((dataset_name, model, f"layer{layer}", filter_name, kmeans)), file)][0]
                labels = np.load(f'{save_path}clustering/{clustering_file}')
            except:
                labels = [0 for i in range(len(embeddings))]
            label_colors = [color_dict[label] for label in labels]
            
            s = 0.0005 if len(labels)> 10000 else 0.05
            
            
            pca, explained_variance_ratio_ = PCA_transform(embeddings)
            
            
            
            
            # scatter plot
            png_axs[plot_row, plot_col].scatter(pca[:,0], pca[:,1], s=s, c = label_colors)
            
            xlim1, xlim2 = png_axs[plot_row, plot_col].get_xlim()
            ylim1, ylim2 = png_axs[plot_row, plot_col].get_ylim()

            xlim1min = min(xlim1min, xlim1)
            xlim2max = max(xlim2max, xlim2)
            ylim1min = min(ylim1min, ylim1)
            ylim2max = max(ylim2max, ylim2)
            
            
            if plot_row == 0:
                
                title_filter_name = "noun-synsets" if filter_name == "noun-synsets-strat" else filter_name
                title = " ".join((dataset_name, title_filter_name)) if title_filter_name != "None" else dataset_name
                png_axs[plot_row, plot_col].set_title(title, fontfamily= "serif")
            
            if plot_col == 0:
                png_axs[plot_row, plot_col].set_ylabel(f"layer {layer}", fontfamily= "serif")
            
            k = max(labels)+1
            png_axs[plot_row, plot_col].text(0.81,0.04, f"k={k}", 
                                             transform = png_axs[plot_row,plot_col].transAxes)
            
            plot_row += 1
            
        
        # adjusting ax limits to be identical per dataset
        for plot_row in range(0,plot_rows):
            png_axs[plot_row, plot_col].set_xlim(xlim1min, xlim2max)
            png_axs[plot_row, plot_col].set_ylim(ylim1min, ylim2max)
        
        plot_col += 1
        
    time = datetime.now().strftime("%m-%d-%H-%M")
    png.savefig(f'{save_path}plot/figure1_clusters_{model}_{time}.png', bbox_inches = "tight")
    plt.show()

        

## Cluster Assignment Plot with best K

In [ ]:
models = ["bert-base-uncased", "microsoft-deberta-base"] # splitted in two plots
layers = [6, 12]  #[0, 3, 6, 9, 12]

datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", "cctweets-random", "cctweets-activist"]

plot_rows = len(layers)
plot_cols = len(datasets)


plt.rc('axes', labelsize=12)
plt.rc('axes', titlesize=12)  
plt.rc("font", size = 10)



for model in models:
      
    png, png_axs = plt.subplots(plot_rows,plot_cols, figsize = (3.5*plot_cols,3.5*plot_rows))
    
    plot_col = 0
    
    ### cols
    for dataset in datasets:
        
        try:
            dataset_name, filter_name = dataset.split("_")
        except:
            dataset_name = dataset
            filter_name = "None"
            
        plot_row = 0

        
        df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)
        
            
        filter_mask = np.arange(0, len(df)) if filter_name == "None" else np.where(df[filter_name])
            
        
        ### rows
        for layer in layers:
            
            
            # loading and filtering embeddings
            embedding_file =  [file for file in os.listdir(f'{save_path}embedding/') if re.match("_".join((dataset_name, model, f"layer{layer}")), file)][0]
            embeddings = np.load(f'{save_path}embedding/{embedding_file}')[filter_mask]
            
            
            # loading the cluster labels
            try:
                clustering_file = [file for file in os.listdir(f'{save_path}clustering/') if re.match("_".join((dataset_name, model, f"layer{layer}", filter_name, "best")), file)][0]
                labels = np.load(f'{save_path}clustering/{clustering_file}')
            except:
                labels = [0 for i in range(len(embeddings))]
            label_colors = [color_dict[label] for label in labels]
            
            s = 0.0005 if len(labels)> 10000 else 0.05
            
            
            pca, explained_variance_ratio_ = PCA_transform(embeddings)
            
            
            
            # scatter plot
            png_axs[plot_row, plot_col].scatter(pca[:,0], pca[:,1], s=s, c = label_colors)
            
            xlim1, xlim2 = png_axs[plot_row, plot_col].get_xlim()
            ylim1, ylim2 = png_axs[plot_row, plot_col].get_ylim()

            xlim1min = min(xlim1min, xlim1)
            xlim2max = max(xlim2max, xlim2)
            ylim1min = min(ylim1min, ylim1)
            ylim2max = max(ylim2max, ylim2)
            
            
            if plot_row == 0:
                
                title_filter_name = "noun-synsets" if filter_name == "noun-synsets-strat" else filter_name
                title = " ".join((dataset_name, title_filter_name)) if title_filter_name != "None" else dataset_name
                png_axs[plot_row, plot_col].set_title(title, fontfamily= "serif")
            
            if plot_col == 0:
                png_axs[plot_row, plot_col].set_ylabel(f"layer {layer}", fontfamily= "serif")
            
            k = max(labels)+1
            png_axs[plot_row, plot_col].text(0.81,0.04, f"k={k}", 
                                             transform = png_axs[plot_row,plot_col].transAxes)
            
            plot_row += 1
            
        
        # adjusting ax limits to be identical per dataset
        for plot_row in range(0,plot_rows):
            png_axs[plot_row, plot_col].set_xlim(xlim1min, xlim2max)
            png_axs[plot_row, plot_col].set_ylim(ylim1min, ylim2max)
        
        plot_col += 1
        
    time = datetime.now().strftime("%m-%d-%H-%M")
    png.savefig(f'{save_path}plot/{model}_{time}.png', bbox_inches = "tight")
    plt.show()

        